# Experiment 1 — does chaining transformations run anything?

Keep the Spark UI open in another tab: http://localhost:4040 → **Jobs**. Refresh it after every cell.

Each **PREDICT** prompt is answered in chat *before* the cell is run.

In [1]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("m1-exp1-lazy").getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/16 18:07:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Step 1 — read two Parquet datasets

**PREDICT:** after this cell, how many jobs does the Jobs tab show?

In [2]:
orders = spark.read.parquet("/opt/data/m1/orders")
customers = spark.read.parquet("/opt/data/m1/customers")

## Step 2 — chain five transformations

**PREDICT:** after this cell, how many jobs? How long does the cell take?

In [3]:
big = orders.filter(F.col("amount") > 100)
big = big.withColumn("amount_eur", F.col("amount") * 0.92)
by_customer = big.groupBy("customer_id").count()
joined = by_customer.join(customers, "customer_id")
top = joined.select("customer_id", "country", "count")

## Step 3 — a typo in a column name

**PREDICT:** what happens when this cell runs — nothing, an error now, or an error later?

In [4]:
bad = orders.select(F.col("amout"))

{"ts": "2026-09-16 18:14:17.268", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `amout` cannot be resolved. Did you mean one of the following? [`amount`, `shop_id`, `order_id`, `order_date`, `customer_id`]. SQLSTATE: 42703", "context": {"file": "line 1 in cell [4]", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o51.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `amout` cannot be resolved. Did you mean one of the following? [`amount`, `shop_id`, `order_id`, `order_date`, `customer_id`]. SQLSTATE: 42703;\n'Project ['amout]\n+- Relation [order_id#0L,customer_id#1,shop_id#2,amount#3,order_date#4] parquet\n\n\tat org.apache.spark.sql.errors.QueryCompilationErrors$.unresolvedAttr

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `amout` cannot be resolved. Did you mean one of the following? [`amount`, `shop_id`, `order_id`, `order_date`, `customer_id`]. SQLSTATE: 42703;
'Project ['amout]
+- Relation [order_id#0L,customer_id#1,shop_id#2,amount#3,order_date#4] parquet


## Step 4 — one action

**PREDICT:** how many jobs appear now? Open the job: how many stages, and how many tasks in each?

In [5]:
top.count()

10000

## Step 5 — a second action on the same DataFrame

**PREDICT:** how many *new* jobs? Does Spark redo all of the work from Step 4?

In [6]:
top.show(5)

+-----------+-------+-----+
|customer_id|country|count|
+-----------+-------+-----+
|       2366|     SE|  151|
|       6336|     DE|  156|
|       3997|     NL|  151|
|       7982|     SE|  180|
|       9427|     ES|  174|
+-----------+-------+-----+
only showing top 5 rows


## Step 6 — explain

**PREDICT:** is `explain()` an action? How many new jobs?

In [7]:
top.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [customer_id#1, country#6, count#10L]
   +- BroadcastHashJoin [customer_id#1], [customer_id#5], Inner, BuildRight, false
      :- HashAggregate(keys=[customer_id#1], functions=[count(1)])
      :  +- Exchange hashpartitioning(customer_id#1, 200), ENSURE_REQUIREMENTS, [plan_id=359]
      :     +- HashAggregate(keys=[customer_id#1], functions=[partial_count(1)])
      :        +- Project [customer_id#1]
      :           +- Filter ((isnotnull(amount#3) AND (amount#3 > 100.0)) AND isnotnull(customer_id#1))
      :              +- FileScan parquet [customer_id#1,amount#3] Batched: true, DataFilters: [isnotnull(amount#3), (amount#3 > 100.0), isnotnull(customer_id#1)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/opt/data/m1/orders], PartitionFilters: [], PushedFilters: [IsNotNull(amount), GreaterThan(amount,100.0), IsNotNull(customer_id)], ReadSchema: struct<customer_id:int,amount:double>
      +- BroadcastExc

## Stop before opening the next notebook

In [8]:
spark.stop()